# Core

> Load API keys from `.env`, count tokens, and read the packaged prompts

In [ ]:
#| default_exp core

In [ ]:
#| export
import os
from pathlib import Path
from fastcore.all import *
import tiktoken

In [ ]:
#| export
def _load_dotenv():
    "Load `.env` from the project root, unless the environment sets `ANTHROPIC_API_KEY` or `MISTRAL_API_KEY`"
    if os.getenv('ANTHROPIC_API_KEY') or os.getenv('MISTRAL_API_KEY'): return
    try:
        from dotenv import load_dotenv
        env_path = Path(__file__).parent.parent / '.env'
        if env_path.exists():
            load_dotenv(env_path)
    except (ImportError, NameError): pass  # NameError: a notebook has no `__file__`

In [ ]:
#| export
_load_dotenv()

Importing `iomeval.core` loads API keys from the `.env` file in the project root, the folder above the `iomeval` package. The `extract`, `mapper`, `pipeline` and `curator` modules import `iomeval.core`, and importing any of them loads the keys too. Variables already in your environment win over the file. If your environment sets `ANTHROPIC_API_KEY` or `MISTRAL_API_KEY`, `iomeval.core` skips the file entirely, including the other key. Without the file or `python-dotenv`, it loads nothing.


In [ ]:
#| export
def n_tokens(
    text:str,         # Text to count tokens in
    model:str='gpt-4' # OpenAI model whose tiktoken encoding to use
    ) -> int:         # Number of tokens
    "Count the tokens in `text` with tiktoken"
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [ ]:
#| eval: false
txt = "This is my test text"
n_tokens(txt)

5

Anthropic models use a different tokenizer. Treat `n_tokens` as an estimate of their token count.


In [ ]:
#| export
def load_prompt(
    name:str,               # Prompt file name without `.md`, such as `'srf_ccps'`
    path:Path|str|None=None # Folder of prompt files; `None` uses the prompts shipped with the package
    ) -> str:               # Prompt text
    "Read the prompt `name` from `path`"
    if path is None:
        try: path = Path(__file__).parent / 'files' / 'prompts'
        except NameError: path = Path('files/prompts')  # A notebook has no `__file__`
    return (Path(path)/f'{name}.md').read_text()

In [ ]:
#| eval: false
print(load_prompt('srf_ccps')[:500])

### ROLE AND PURPOSE 
You are a triage assistant supporting evaluation synthesis specialists at the International Organization for Migration (IOM). Your role is to help prioritize which evaluation reports warrant in-depth review for each of the IOM Strategic Result Framework (SRF) Cross-cutting Priorities. 

Important context about your role: 
- You are suggesting relevance, not making definitive judgments 
- Human specialists will review your assessments and make final decisions 
- Your scores 


The package ships five prompts. `identify_core_sections` uses `select_sections`. The framework mappings use `srf_enablers`, `srf_ccps`, `gcms` and `srf_outputs`. Pass `path` to use your own versions.
